# Preferred Stocks — a quantitative teardown 🔬
### Real total-return tapes · full vs downside beta · HAC + block-bootstrap · crash co-movement

![Signal: Real](https://img.shields.io/badge/Signal-Real-2ea44f?style=flat-square)
![Tradability: Mirage](https://img.shields.io/badge/Tradability-Mirage-c0392b?style=flat-square)
![Bond-like safety?: Busted](https://img.shields.io/badge/Bond--like_safety%3F-Busted-8b949e?style=flat-square)

The deep companion to the [notebook for the curious](01_for_the_curious.ipynb) — *same seven beats, every claim now carrying its standard error.* We test the three things the preferred-stock label sells — a **fat yield** (real, but ≈ a Treasury's total return), **bond-like safety** (mirage), and a **crash cushion** (busted) — and pin down *what PFF actually is* via its beta to stocks vs bonds, in the body and in the tail.

> ⚠️ **Not investment advice.** PFF/SPY/IEF daily, **total-return** adjusted (`preferred_stocks.data`, yfinance `auto_adjust=True`); betas via OLS with a HAC *t* and a circular block bootstrap on the difference. PFF inception bounds the window at 2007-03-30. Sources in [`docs/references.md`](../docs/references.md), reproducible run in [`docs/results.md`](../docs/results.md).
>
> 💡 **The `💡 In plain words` notes** translate each result back into intuition.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))          # study root (preferred_stocks/)
sys.path.insert(0, os.path.abspath("../../.."))    # repo root (quantlab/)
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (10, 5.5)
import numpy as np, pandas as pd
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
from preferred_stocks import data, strategy

AS_OF = "2026-05-31"
# Try the REAL total-return tapes; fall back to the offline synthetic control if the
# network is blocked. Every cell prints which TAPE it is showing.
try:
    prices = data.load_real(("PFF", "SPY", "IEF")).loc[:AS_OF]
    cols = {"PFF": "PFF", "SPY": "SPY", "BND": "IEF"}   # bond proxy label
    TAPE = "REAL"
except Exception as e:
    print("network/cache miss -> SYNTHETIC control tape:", type(e).__name__)
    prices, _truth = data.synthetic_three_asset(pref_beta=0.90, seed=338)
    cols = {"PFF": "PFF", "SPY": "SPY", "BND": "BND"}
    TAPE = "SYNTHETIC"

PFF, SPY, BND = cols["PFF"], cols["SPY"], cols["BND"]
rets = strategy.to_returns(prices)
BANNER = ("REAL total-return tape (PFF/SPY/IEF, 2007-2026)" if TAPE == "REAL"
          else "SYNTHETIC control tape (pref_beta=0.90) -- NOT market data")
print(f"[{TAPE}] {BANNER}")
print(f"panel: {len(prices):,} rows  {prices.index[0].date()} -> {prices.index[-1].date()}  fingerprint={data.fingerprint(prices)}")
for c in (PFF, SPY, BND):
    s = strategy.stats(rets[c])
    print(f"  {c}: CAGR {s['cagr']*100:6.2f}%  vol {s['vol']*100:5.1f}%  Sharpe {s['sharpe']:.3f}  maxDD {s['max_dd']*100:6.1f}%")


[REAL] REAL total-return tape (PFF/SPY/IEF, 2007-2026)
panel: 4,822 rows  2007-03-30 -> 2026-05-29  fingerprint=225bc4dab397
  PFF: CAGR   3.88%  vol  18.7%  Sharpe 0.298  maxDD  -65.5%
  SPY: CAGR  11.16%  vol  19.7%  Sharpe 0.635  maxDD  -55.2%
  IEF: CAGR   3.26%  vol   7.0%  Sharpe 0.495  maxDD  -23.9%


## Verdict, up front

| Axis | Stamp | Why |
|---|---|---|
| **Signal** | `REAL` | Beta of PFF to **stocks** **+0.55** (HAC *t* **+5.58**), to **bonds** -0.206; downside beta to stocks **0.98** vs 0.1 to bonds; fell in 7/7 crashes. |
| **Tradability** | `MIRAGE` | Return is bond-like (CAGR **3.88%** vs IEF **3.26%**) but risk is equity-grade (vol **18.7%**, maxDD **-65.5%**, deeper than SPY). Not a safe income sleeve. |
| **Bond-like safety?** | `BUSTED` | In the GFC PFF lost -63.7% (worse than SPY) while bonds gained +20.2%. Seniority ≠ a bond. |

> 💡 **In plain words:** PFF is **equity-grade crash risk wearing a bond's coupon**. The risk is statistically real; the safety is the part that isn't there.

## 1 · The claim, steelmanned

- **H₁ (yield):** PFF's fixed coupon delivers materially more total return than bonds.
- **H₂ (safety):** PFF's risk (vol, drawdown, crash co-movement) is bond-like, not equity-like.
- **H₃ (cushion):** PFF holds up — or at least doesn't fall hard — when equities crash.

H₁ is **weak** (return ≈ a Treasury's), and H₂/H₃ are **false** — PFF loads on equities, and the loading intensifies exactly in the tail.

## 2 · So what? — what rides on each answer

If H₂ holds, PFF belongs in the *safe* sleeve and the 2008 result is an aberration. If PFF is equity-in-disguise, every income investor filing it under fixed income is mis-stating their true equity exposure — and discovers it in the next crash. This is the same 'is this asset what the brochure says?' question the desk asked of gold ([Study 69](../../69-safe-haven/)) and REITs ([Study 207](../../207-reits-diversifier/)).

## 3 · How we'd know — the protocol

Return/vol/drawdown on the total-return tape · full-sample beta of PFF to SPY and to IEF, each with a **HAC *t*** on the OLS-slope influence series · **downside** beta on the worst 10% of equity days · crash co-movement in every >10% equity drawdown · a **circular block bootstrap** CI on the downside-beta *difference* (to-stocks − to-bonds).

## 4 · The teardown

Headline stats — return is bond-like, risk is not:

In [2]:
tbl = pd.DataFrame({
  'CAGR %':  [strategy.stats(rets[c])['cagr']*100 for c in (PFF, SPY, BND)],
  'Vol %':   [strategy.stats(rets[c])['vol']*100 for c in (PFF, SPY, BND)],
  'Sharpe':  [strategy.stats(rets[c])['sharpe'] for c in (PFF, SPY, BND)],
  'MaxDD %': [strategy.stats(rets[c])['max_dd']*100 for c in (PFF, SPY, BND)],
}, index=[PFF, f'{SPY} (equity)', f'{BND} (bonds)'])
print(f'[{TAPE}] tape')
tbl.round(2)

[REAL] tape


,CAGR %,Vol %,Sharpe,MaxDD %
PFF,3.880,18.660,0.300,-65.550
SPY (equity),11.160,19.720,0.640,-55.190
IEF (bonds),3.260,6.970,0.490,-23.920


**Who does PFF move with?** OLS beta to stocks and to bonds, each with a HAC *t* on the slope's influence series (mean of the influence series = the OLS slope).

In [3]:
def hac_beta_t(p, x):
    p = p.to_numpy(float); x = x.to_numpy(float); xc = x - x.mean()
    infl = (p - p.mean()) * xc / (xc @ xc) * len(xc)
    return float(infl.mean()), strategy.hac_tstat(infl)
b_spy, t_spy = hac_beta_t(rets[PFF], rets[SPY])
b_bnd, t_bnd = hac_beta_t(rets[PFF], rets[BND])
print(f'[{TAPE}]  beta PFF~{SPY} = {b_spy:+.3f}  (HAC t {t_spy:+.2f})')
print(f'[{TAPE}]  beta PFF~{BND} = {b_bnd:+.3f}  (HAC t {t_bnd:+.2f})')
print(f'downside beta to {SPY} (worst 10% days) = {strategy.downside_beta(rets[PFF], rets[SPY]):.3f}')
print(f'downside beta to {BND} (worst 10% days) = {strategy.downside_beta(rets[PFF], rets[BND]):.3f}')

[REAL]  beta PFF~SPY = +0.550  (HAC t +5.58)
[REAL]  beta PFF~IEF = -0.206  (HAC t -2.02)


downside beta to SPY (worst 10% days) = 0.977
downside beta to IEF (worst 10% days) = 0.104


> 💡 **In plain words:** on the real tape the beta to stocks is **+0.55** with a HAC *t* of **+5.58** — far past the *t*=2 bar — while the beta to bonds is *negative*. And on the worst equity days the stock-beta climbs to **~0.98**: PFF inherits almost the *entire* equity move precisely when 'safety' was the point.

**Block-bootstrap on the difference** — is PFF's downside beta to stocks reliably above its downside beta to bonds? (Tail-conditioned betas are noisy, so we resample jointly in blocks and report the CI and the win-rate.)

In [4]:
boot = strategy.bootstrap_downside_beta_diff(rets[PFF], rets[SPY], rets[BND], block=21, n_boot=2000, seed=338)
print(f'[{TAPE}] downside-beta diff (to-{SPY} minus to-{BND}) = {boot["point"]:+.3f}')
print(f'  95% block-bootstrap CI [{boot["ci95"][0]:+.3f}, {boot["ci95"][1]:+.3f}]')
print(f'  PFF more equity-like in {boot["frac_spy_wins"]*100:.0f}% of resamples')

[REAL] downside-beta diff (to-SPY minus to-IEF) = +0.873
  95% block-bootstrap CI [-0.191, +2.380]
  PFF more equity-like in 93% of resamples


> 💡 **In plain words:** the point difference is large and positive and PFF is the more equity-like asset in the large majority of resamples. The *certified* result rests on the full-sample beta (HAC *t* +5.58); the tail-beta bootstrap is the corroborating picture — honestly, its CI is wide because crash days are few.

**Crash co-movement** — every >10% equity drawdown, PFF and bonds over the same window:

In [5]:
eps = strategy.equity_drawdowns(rets[[SPY, PFF, BND]], SPY, thresh=-0.10)
rows = [{'peak':e['peak'].date(),'trough':e['trough'].date(),
         f'{SPY} %':e['stock_loss']*100, f'{PFF} %':e['others'][PFF]*100, f'{BND} %':e['others'][BND]*100} for e in eps]
dd = pd.DataFrame(rows)
print(f'[{TAPE}] PFF fell in {(dd[f"{PFF} %"]<0).sum()}/{len(dd)}; bonds rose in {(dd[f"{BND} %"]>0).sum()}/{len(dd)}')
dd.round(1)

[REAL] PFF fell in 7/7; bonds rose in 5/7


,peak,trough,SPY %,PFF %,IEF %
0,2007-10-09,2009-03-09,-55.200,-63.700,20.200
1,2015-07-20,2016-02-11,-13.000,-5.000,7.400
2,2018-01-26,2018-02-08,-10.100,-2.400,-1.200
3,2018-09-20,2018-12-24,-19.300,-8.100,3.400
4,2020-02-19,2020-03-23,-33.700,-30.800,6.400
5,2022-01-03,2022-10-12,-24.500,-18.500,-15.400
6,2025-02-19,2025-04-08,-18.800,-7.600,2.700


## 5 · The verdict

Signal `REAL` (beta to stocks +0.55, HAC *t* +5.58; downside beta ~0.98; fell 7/7 crashes). Tradability `MIRAGE` (bond-grade return 3.88% for equity-grade risk: vol 18.7%, maxDD -65.5%). Bond-like-safety? `BUSTED` (GFC PFF -63.7% vs bonds +20.2%).

## 6 · Could you trade it?

Capacity and liquidity are fine in calm markets — PFF trades millions of shares a day. The reservations are structural, not frictional: (1) preferreds are **perpetual, callable and deeply subordinated**, so they carry credit-spread *and* equity-tail exposure; (2) the index is **bank- and insurer-heavy**, a concentrated bet on the sector that fails first in a crisis; (3) the coupon is compensation for that risk, not a free yield premium. As a yield *enhancer* understood as risk: defensible. As the *safe* income sleeve: a mirage.

## 7 · Going further

- Decompose PFF total return into **coupon vs price**: how much of the 3.9%/yr CAGR is the coupon being eroded by capital losses in 2008/2020/2022?
- Re-run on a **non-financials-tilted** preferred index (PFXF) — does removing the bank concentration soften the 2008 beta, or is it the security structure, not the sector?
- Add a **convertible-bond** arm: another 'bond-equity hybrid' whose label hides its true exposure.